## initialization

In [23]:
import os,sys,subprocess,glob,cftime,importlib,pickle,itertools
from datetime import datetime
import xarray as xr
import numpy as np
import pandas as pd
import scipy

sys.path.append('../REA_with_CESM2')
from ensembles.ensemble_GKLT import ensemble_GKLT,get_weight_for_selection

sys.path.append('../REA_heat_wEU_JJA')
from experiment_configuration.experiment import experiment

def import_from(module, name):
    module = __import__(module, fromlist=[name])
    return getattr(module, name)

import warnings
warnings.filterwarnings('ignore')

from sup.sup_plotting import *
from sup.merged_ensembles import *

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
data_dir = '' # sepcify where the rare event sampling output is stored
climates = ['ssp370-2025', 'piControl']

In [ ]:
# load data prepared in 0_main_analysis.ipynb
with open('pickles/REA_heat_1D_stats.pkl', 'rb') as fl:
    ensembles = pickle.load(fl)

In [26]:
rt = 100
ens = ensembles['piControl-LE-all']
threshold_anomaly = weighted_quantile(ens['data']['tas_anom'].mean('time'), (1 - 1/rt), sample_weight=ens['data']['weight'])
threshold_anomaly

2.2122549116517263

In [27]:
def weighted_expected_value(
        tas, 
        weight, 
        threshold,
        x,
        ):
    return ((tas >= threshold) * x * weight).sum('sim') / ((tas >= threshold) * weight).sum('sim')

In [28]:
def preprocess(nc,var):
    if nc.lon[0] == 0:
        nc = nc.roll(lon=144, roll_coords=True)
    nc = nc.assign_coords(lon=(nc.lon + 180) % 360 - 180).sel(dict(lat=slice(-20,90))).load()
    if var == 'tos':
        if nc['tos'].max() > 273:
            nc['tos'] -= 273.15
            nc['tos'].values[nc['tos'] < -200] = np.nan
    nc[var] = nc[var].mean('time')
    return nc

## Compute composites

### REA

In [ ]:

for var in ['pr', 'tos', 'mrsos', 'tas', 'eke500', 'zg500','stream500','ua500','va500','ua200']:
    print(var)
    var_name_in_file = var

    for r,climate in enumerate(climates):
        clim_mean = ensembles[f'{climate}-LE-all']['data']['tas_absolute'].values.mean()

        l = []
        for i in range(1,6):
            ens_name = f"{climate}-x{i}"
            try:
                with xr.open_mfdataset(f"{data_dir}/{ens_name}/meta/obs/*/*", concat_dim='sim', combine='nested') as nc:
                    tas = nc['obs'].mean('time').load()

                with xr.open_dataset(f"{data_dir}/{ens_name}/meta/weight_season_{ens_name}.nc") as nc:
                    weight = nc['weight'].load()

                with xr.open_mfdataset(f"{data_dir}/{ens_name}/mon/*/{var}/*/*", concat_dim='sim', combine='nested') as nc:
                    tmp = preprocess(nc,var_name_in_file)[var_name_in_file].load() 
                    l.append(
                        weighted_expected_value(
                            tas=tas, 
                            weight=weight, 
                            threshold=threshold_anomaly + clim_mean, 
                            x=tmp
                        )
                    )
                    attrs = nc.attrs
            except:
                print(ens_name, 'fail')
        y = xr.concat(l, dim='sim').mean('sim')
        xr.Dataset({var:y}).to_netcdf(f'average_maps/{var}_{climate}_rea_100.nc')


pr
tos
mrsos
ssp370-2025-x1 fail
ssp370-2025-x2 fail
piControl-x1 fail
piControl-x2 fail
tas
eke500
ssp370-2025-x1 fail
piControl-x1 fail
zg500
stream500
ssp370-2025-x1 fail
piControl-x1 fail
ua500
ssp370-2025-x1 fail
piControl-x1 fail
va500
ssp370-2025-x1 fail
piControl-x1 fail
ua200
ssp370-2025-x1 fail
piControl-x1 fail


### REA June and August

In [29]:
def preprocess_monthly(nc,var):
    if nc.lon[0] == 0:
        nc = nc.roll(lon=144, roll_coords=True)
    nc = nc.assign_coords(lon=(nc.lon + 180) % 360 - 180).sel(dict(lat=slice(-20,90))).load()
    if var == 'tos':
        if nc['tos'].max() > 273:
            nc['tos'] -= 273.15
            nc['tos'].values[nc['tos'] < -200] = np.nan
    nc[var] = nc[var]
    return nc

In [34]:

for var in ['tos']:
    print(var)
    var_name_in_file = var

    for r,climate in enumerate(climates):
        clim_mean = ensembles[f'{climate}-LE-all']['data']['tas_absolute'].values.mean()

        for month_i,month in [
            (0, "jun"),
            (1, "aug"),
        ]:
            l = []
            for i in range(1,6):
                ens_name = f"{climate}-x{i}"
                with xr.open_mfdataset(f"{data_dir}/{ens_name}/meta/obs/*/*", concat_dim='sim', combine='nested') as nc:
                    tas = nc['obs'].mean('time').load()

                with xr.open_dataset(f"{data_dir}/{ens_name}/meta/weight_season_{ens_name}.nc") as nc:
                    weight = nc['weight'].load()

                with xr.open_mfdataset(f"{data_dir}/{ens_name}/mon/*/{var}/*/*", concat_dim='sim', combine='nested') as nc:
                    tmp = preprocess_monthly(nc,var_name_in_file)[var_name_in_file]
                    tmp = tmp[:,month_i].load() 

                    l.append(
                        weighted_expected_value(
                            tas=tas, 
                            weight=weight, 
                            threshold=threshold_anomaly + clim_mean, 
                            x=tmp
                        )
                    )
                    attrs = nc.attrs

            y = xr.concat(l, dim='sim').mean('sim')
            xr.Dataset({var:y}).to_netcdf(f'average_maps/{var}_{climate}_rea_100_{month}.nc')


tos


### SST STD map for initial

In [35]:
for climate in climates:
    with xr.open_mfdataset(f"{data_dir}/{climate}-initial/mon/*/tos/*/*", concat_dim='sim', combine='nested') as nc:
        tmp = preprocess(nc,'tos')['tos'].load() 
        xr.Dataset({'tos':tmp.std('sim')}).to_netcdf(f'std_maps/tos_{climate}_initial_std.nc')
        xr.Dataset({'tos':tmp.mean('sim')}).to_netcdf(f'average_maps/tos_{climate}_initial_average.nc')

In [37]:
for climate in climates:
    with xr.open_mfdataset(f"{data_dir}/{climate}-initial/mon/*/tos/*/*", concat_dim='sim', combine='nested') as nc:
        tos = preprocess_monthly(nc,'tos')['tos']
        for month_i,month in [
            (0, "jun"),
            (1, "aug"),
        ]:
            tmp = tos[:,month_i].load() 
            xr.Dataset({'tos':tmp.std('sim')}).to_netcdf(f'std_maps/tos_{climate}_initial_std_{month}.nc')
            xr.Dataset({'tos':tmp.mean('sim')}).to_netcdf(f'average_maps/tos_{climate}_initial_average_{month}.nc')


### REA dry

In [ ]:
climate = 'ssp370-2025'
rt = 100
ens = ensembles['piControl-LE-all']
clim_mean = ensembles[f'{climate}-LE-all']['data']['tas_absolute'].values.mean()

for var in ['tas', 'zg500']:
    print(var)
    var_name_in_file = var
    l = []
    for ens_name in [
        "ssp370-2025-dry-x6",
        "ssp370-2025-dry-x9",
        "ssp370-2025-dry-x11",
        "ssp370-2025-wet-x7",
        "ssp370-2025-wet-x8",
        "ssp370-2025-wet-x12",
    ]:
        try:
            with xr.open_mfdataset(f"{data_dir}/{ens_name}/meta/obs/*/*", concat_dim='sim', combine='nested') as nc:
                tas = nc['obs'].mean('time').load()

            with xr.open_dataset(f"{data_dir}/{ens_name}/meta/weight_season_{ens_name}.nc") as nc:
                weight = nc['weight'].load()

            with xr.open_mfdataset(f"{data_dir}/{ens_name}/mon/*/{var}/*/*", concat_dim='sim', combine='nested') as nc:
                tmp = preprocess(nc,var_name_in_file)[var_name_in_file].load() 
                l.append(
                    weighted_expected_value(
                        tas=tas, 
                        weight=weight, 
                        threshold=threshold_anomaly + clim_mean, 
                        x=tmp
                    )
                )
                attrs = nc.attrs
        except:
            print(ens_name, 'fail')
    y = xr.concat(l, dim='sim').mean('sim')
    xr.Dataset({var:y}).to_netcdf(f'average_maps/{var}_ssp370-2025-dry_rea_100.nc')


tas
zg500


### LE

In [ ]:
data_path_LE = "" # specify where LE data is stored

In [ ]:
for climate,year_start,year_stop in zip(climates, [2020,1850], [2030,1900]):
    print(climate)

    for var, var_original, folder_name in [
        ('tas', 'TREFHT', 'TREFHT/day_raw'),
        ('zg500', 'Z500', 'NEW_Z500/daily_raw'),
        ('pr', 'PRECT', 'PRECT/day_raw'),
        ('mrsos', 'SOILWATER_10CM', 'SOILWATER_10CM/day_raw'),
        ('tos', 'SST', 'SST/mon_raw'),
        ]:
        out_file = f'average_maps/{var}_{climate}_LE_average.nc'
        if os.path.isfile(out_file) == False or os.path.isfile(out_file.replace('average','std')) == False or True:
            potential_files = glob.glob(f'{data_path_LE}/{folder_name}/*.nc')
            l = []
            for fl in tqdm(sorted(potential_files)):
                yl,yh = (int(s[:4]) for s in fl.split('.')[-2].split('-'))
                if yl <= year_stop and yh >= year_start:
                    with xr.open_dataset(fl) as nc:
                        nc = nc.rename({var_original:var})
                        nc = preprocess(nc, var)
                        y = nc[var][(nc.time.dt.year >= year_start) & (nc.time.dt.year <= year_stop) & np.isin(nc.time.dt.month.values, [6,7,8])].resample(time='1Y').mean().load()
                        l.append(y)
            nc = xr.Dataset({var:xr.concat(l, dim='time').mean('time')})
            nc.to_netcdf(out_file)
            nc = xr.Dataset({var:xr.concat(l, dim='time').std('time')})
            nc.to_netcdf(out_file.replace('average','std'))


In [ ]:
for climate,year_start,year_stop in zip(climates, [2020,1850], [2030,1900]):
    files = sorted(glob.glob(f'daily/TREFHT/*'))
    simulation_names = np.array([s.split('/')[-1][:-3] for s in files])
    with xr.open_mfdataset(files) as nc:
        x = nc['TREFHT'][:, (nc.time.dt.year >= year_start) & (nc.time.dt.year <= year_stop) & np.isin(nc.time.dt.month.values, [6,7,8]) ].load()
        x -= 273.15
        # transform to anomalies
        x -= float(reference_for_anomalies.loc[climate])


    for var, var_original, folder_name in [
        ('tas', 'TREFHT', 'TREFHT/day_raw'),
        ('zg500', 'Z500', 'NEW_Z500/daily_raw'),
        ('pr', 'PRECT', 'PRECT/day_raw'),
        ('mrsos', 'SOILWATER_10CM', 'SOILWATER_10CM/day_raw'),
        ('tos', 'SST', 'SST/mon_raw'),
        ]:
        for rt in [50,100]:
            print(climate, var, rt)
            out_file = f'average_maps/{var}_{climate}_LE_{rt}.nc'
            if os.path.isfile(out_file) == False or True:
                # identify events
                i_sims, i_years = np.where(x.values.reshape((100,-1,92))[:,:,:-2].mean(axis=2) > float(thresholds.loc['both', rt]))

                l = []
                for sim,year in zip(simulation_names[np.array(i_sims)], np.unique(x.time.dt.year.values)[i_years]):
                    potential_files = glob.glob(f'{data_path_LE}/{folder_name}/*{sim}*')

                    for fl in potential_files:
                        yl,yh = (int(s[:4]) for s in fl.split('.')[-2].split('-'))
                        if yl <= year and yh >= year:
                            with xr.open_dataset(fl) as nc:
                                nc = nc.rename({var_original:var})
                                nc = preprocess(nc, var)
                                y = nc[var][(nc.time.dt.year == year) & np.isin(nc.time.dt.month.values, [6,7,8]) ][:90].mean('time').load()
                                l.append(y)

                nc = xr.Dataset({var: xr.concat(l, dim='time').mean('time').squeeze()})
                nc.to_netcdf(out_file)


## Plots

In [ ]:
def plot_together(extent=None, pixels=False, axes=None, maxabs=2):
    cmap = matplotlib.colormaps["RdBu_r"]
    cmap.set_under("c")   # color for values < vmin
    cmap.set_over("m")  

    for row, col, var,label,a_name, b_name in details:

        a = xr.load_dataarray(f'average_maps/{var}_{a_name}.nc').squeeze().sel(dict(lat=slice(-20,91)))
        b = xr.load_dataarray(f'average_maps/{var}_{b_name}.nc').squeeze().sel(dict(lat=slice(-20,91)))
        b_climate = b_name.split('_')[0]
        b_ens = b_name.split('_')[1]
        st = xr.load_dataarray(f"std_maps/{var}_{b_climate}_{b_ens}_std.nc").squeeze().sel(dict(lat=slice(-20,91)))

        xxx = (a - b.values) / st.values
        xxx = xxx
        ax = axes[row,col]
        ax.coastlines(linewidth=0.3, color='gray')
        if extent is not None:
            ax.set_extent(extent)
        levels = np.linspace(-maxabs,maxabs,24)
        if pixels:
            im = ax.pcolormesh(xxx.lon, xxx.lat, xxx, transform=cartopy.crs.PlateCarree(), vmin=-maxabs, vmax=maxabs, cmap=cmap)
        else:
            im = ax.contourf(xxx.lon, xxx.lat, xxx, transform=cartopy.crs.PlateCarree(), levels=levels, extend='both' , cmap=cmap)

        ax.add_patch(matplotlib.patches.Rectangle(xy=[-4, 44], width=16, height=11, facecolor='none', edgecolor='k', transform=cartopy.crs.PlateCarree()))

        t = ax.annotate(string.ascii_lowercase[col+row*2], xy=(0,1), xycoords="axes fraction", va='center', ha='center', clip_on=False, fontsize=15)
        t.set_bbox(dict(facecolor='w', edgecolor='none'))

        ax.set_title(label)
        #if col == 1:
            #cbar = plt.colorbar(im, ax=axes[row,-1], label=label)
            #cbar.set_ticks(np.linspace(-maxabs,maxabs,6))

    return im

In [ ]:
rt = 100

details = [
    (0, 0, 'tas', 'temperature (pre-ind.)', f'piControl_rea_{rt}', f'piControl_LE_average'),
    (1, 0, 'zg500', 'geopotential height 500hPa (pre-ind.)', f'piControl_rea_{rt}', f'piControl_LE_average'),
    (2, 0, 'pr', 'precipitation (pre-ind.)', f'piControl_rea_{rt}', f'piControl_LE_average'),
    (3, 0, 'mrsos', 'soilmoisture (pre-ind.)', f'piControl_rea_{rt}', f'piControl_LE_average'),
    (4, 0, 'tos', 'SST (pre-ind.)', f'piControl_rea_{rt}', f'piControl_initial_average'),
    
    (0, 1, 'tas', 'temperature (current)', f'ssp370-2025_rea_{rt}', f'ssp370-2025_LE_average'),
    (1, 1, 'zg500', 'geopotential height 500hPa (current)', f'ssp370-2025_rea_{rt}', f'ssp370-2025_LE_average'),
    (2, 1, 'pr', 'precipitation (current)', f'ssp370-2025_rea_{rt}', f'ssp370-2025_LE_average'),
    (3, 1, 'mrsos', 'soilmoisture (current)', f'ssp370-2025_rea_{rt}', f'ssp370-2025_LE_average'),
    (4, 1, 'tos', 'SST (current)', f'ssp370-2025_rea_{rt}', f'ssp370-2025_initial_average'),
    ]



nrows = int(len(details)/2)
ncols = 2
fig,axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*4,nrows*1.5), 
                        subplot_kw={"projection":cartopy.crs.PlateCarree()}, gridspec_kw=dict(width_ratios = [10,10]))

im = plot_together(axes=axes, maxabs=2)
cax = fig.add_axes([0.2, -0.02, 0.6, 0.02])
cbar = plt.colorbar(im, cax=cax, orientation='horizontal', label='difference [$\sigma$]')
cbar.set_ticks(np.linspace(-2,2,6))
plt.tight_layout()
savefig(f'diff_maps/diff_maps_all_{rt}', bbox_inches='tight')
savefig(f'for_paper/diff_maps_all_{rt}', also_png=False, bbox_inches='tight')

In [ ]:
rt = 100

details = [
    (0, 0, 'tos', 'June SST (pre-ind.)', f'piControl_rea_{rt}_jun', f'piControl_initial_average_jun'),
    (0, 1, 'tos', 'June SST (current)', f'ssp370-2025_rea_{rt}_jun', f'ssp370-2025_initial_average_jun'),
    (1, 0, 'tos', 'August SST (pre-ind.)', f'piControl_rea_{rt}_aug', f'piControl_initial_average_aug'),
    (1, 1, 'tos', 'August SST (current)', f'ssp370-2025_rea_{rt}_aug', f'ssp370-2025_initial_average_aug'),
    ]

fig,axes = plt.subplots(nrows=2, ncols=2, figsize=(6,4), 
                        subplot_kw={"projection":cartopy.crs.PlateCarree()}, gridspec_kw=dict(width_ratios = [10,10]))

im = plot_together(extent=[-30,50,20,80], pixels=True, axes=axes, maxabs=2)
cax = fig.add_axes([0.2, -0.02, 0.6, 0.02])
cbar = plt.colorbar(im, cax=cax, orientation='horizontal', label='difference [$\sigma$]')
cbar.set_ticks(np.linspace(-2,2,6))
plt.tight_layout()
savefig(f'diff_maps/diff_map_sst_100_jun_aug', bbox_inches='tight')
savefig(f'for_paper/diff_map_sst_100_jun_aug', also_png=False, bbox_inches='tight')